# ResearchAgent — Part 2: Retriever

**Pipeline stage covered here:** `sub-questions → Retriever → attributed search snippets`

Run `01_setup_and_planner.ipynb` first (or re-run its setup cells here) so
`GROQ_API_KEY` and the `plan()` function are available.

## 1. Install dependencies

In [ ]:
# !pip install -q ddgs

## 2. Retriever

Takes the sub-questions from the Planner and runs a web search for each one.
Every snippet is tagged with its source question and a citation-friendly index
(e.g. `[pricing-1]`) so the Drafter can cite it later and the Critic can check
groundedness.

In [ ]:
from dataclasses import dataclass, field
from ddgs import DDGS


@dataclass
class SearchResult:
    sub_question: str
    snippets: list[dict] = field(default_factory=list)  # [{title, url, body}, ...]


def retrieve(sub_questions: list[str], max_results_per_query: int = 4) -> list[SearchResult]:
    """Search the web for each sub-question.

    Args:
        sub_questions: Output of plan().
        max_results_per_query: How many results to pull per sub-question.

    Returns:
        List of SearchResult, one per sub-question, each holding attributed snippets.
    """
    results = []
    with DDGS() as ddgs:
        for question in sub_questions:
            hits = list(ddgs.text(question, max_results=max_results_per_query))
            snippets = [
                {"title": h.get("title", ""), "url": h.get("href", ""), "body": h.get("body", "")}
                for h in hits
            ]
            results.append(SearchResult(sub_question=question, snippets=snippets))
    return results

## 3. Format for the Drafter

Flattens the search results into a single text block with citation tags like
`[pricing-1]` inline — this is what gets fed into the Drafter's prompt in
Part 3, and it's how the final report is able to cite sources instead of just
asserting facts.

In [ ]:
def format_for_prompt(results: list[SearchResult]) -> str:
    """Flatten retrieved results into a citation-friendly text block for the drafter."""
    blocks = []
    for r in results:
        blocks.append(f"### Sub-question: {r.sub_question}")
        for i, s in enumerate(r.snippets, 1):
            blocks.append(f"[{r.sub_question[:20]}-{i}] {s['title']}\nURL: {s['url']}\n{s['body']}\n")
    return "\n".join(blocks)

## 4. Testing

If you ran Part 1 in this same session, `sub_questions` is already defined —
otherwise this cell generates a fresh sample list so you can test the
Retriever standalone.

In [ ]:
# Reuse sub_questions from Part 1 if it exists, otherwise use a sample set
try:
    test_questions = sub_questions
except NameError:
    test_questions = [
        "What is the pricing model of Notion AI?",
        "What is the pricing model of Mem?",
        "What is the pricing model of Reflect?",
    ]

results = retrieve(test_questions)

for r in results:
    print(f"\n=== {r.sub_question} ===")
    for s in r.snippets:
        print(f"- {s['title']}  ({s['url']})")

In [ ]:
formatted = format_for_prompt(results)
print(formatted[:1500])  # preview — full text can be long